# Tokenization + Model Setup + Training_Model + Evaluation + Model Saving + Sample Prediction

## 2A: Import + Load Cleaned Data

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/clean_tweets')
df

,text,label
0,@VirginAmerica What @dhepburn said.,1
1,@VirginAmerica plus you've added commercials t...,2
2,@VirginAmerica I didn't today... Must mean I n...,1
3,@VirginAmerica it's really aggressive to blast...,0
4,@VirginAmerica and it's a really big bad thing...,0
...,...,...
14635,@AmericanAir thank you we got on a different f...,2
14636,@AmericanAir leaving over 20 minutes Late Flig...,0
14637,@AmericanAir Please bring American Airlines to...,1
14638,"@AmericanAir you have my money, you change my ...",0


In [12]:
# train and test
train_texts, val_texts, train_labels, val_labels = train_test_split(df['text'].tolist(),df['label'].tolist(),test_size=0.2, random_state=46)

hugging face tokeniser expects a list of strings, not a panda series, that's why: tolist() used above

## 2B: Tokenization

1. tokenizer - Converts text into token IDs
2. truncation - Ensures that no text exceeds max token length (e.g., 512 tokens)
3. Makes all tokenized sequences the same length by padding shorter ones
4. attention_mask - Tells model which parts are real (1) or padding (0)


In [19]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

#tokenize the data
train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)

## 2C: Create Dataset Objects (HuggingFace Compatible)

In [20]:
import torch

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)

    def __getitem__(self,idx):
        item = {key: torch.tensor(val[idx]) for key,val in self.encodings.items()}#transformer only accepts items in tensor
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = TweetDataset(train_encodings, train_labels)
val_dataset = TweetDataset(val_encodings, val_labels)

## 3A: Load the Model

In [21]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels = 3 ) #we have 3 labels
#You just loaded a pre-trained DistilBERT model, with a classification head for 3 classes.

Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertForSequenceClassification: ['vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_transform.weight', 'vocab_transform.bias', 'vocab_layer_norm.weight']
- This IS expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier.bias', 'pre_classifier.weight', 'classifier.weight', 'classifier.

## 3B: Define Training Arguments

In [22]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',             # where to save model
    num_train_epochs=3,                 # you can increase this later
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    evaluation_strategy='epoch',       # eval after every epoch
    save_strategy='epoch',
    logging_dir='./logs',              # for TensorBoard
    load_best_model_at_end=True,
    metric_for_best_model='accuracy'
)

## 3C: Define Metrics

In [23]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {
        'accuracy': acc,
        'f1': f1
    }
# This gives meaningful evaluation, not just accuracy — F1 score is crucial in imbalanced datasets.

## 3D: Train with HuggingFace Trainer

In [24]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.530300,0.420041,0.849044,0.839694
2,0.356400,0.445201,0.842896,0.840184
3,0.159600,0.569637,0.848361,0.845925


TrainOutput(global_step=2196, training_loss=0.3159980634956846, metrics={'train_runtime': 1333.932, 'train_samples_per_second': 26.34, 'train_steps_per_second': 1.646, 'total_flos': 590897927767680.0, 'train_loss': 0.3159980634956846, 'epoch': 3.0})

# Step 4: Evaluate the Trained Model

## 4A. Evaluate and Print Accuracy & F1

In [25]:
eval_results = trainer.evaluate()
print(eval_results)

{'eval_loss': 0.42004093527793884, 'eval_accuracy': 0.8490437158469946, 'eval_f1': 0.8396942870783237, 'eval_runtime': 27.9435, 'eval_samples_per_second': 104.783, 'eval_steps_per_second': 1.646, 'epoch': 3.0}


# Step 5: Save the Model and Tokenizer

## 5A. Save the Trained Model

In [26]:
model.save_pretrained("../models/distilbert-sentiment")
tokenizer.save_pretrained("../models/distilbert-sentiment")

('../models/distilbert-sentiment/tokenizer_config.json',
 '../models/distilbert-sentiment/special_tokens_map.json',
 '../models/distilbert-sentiment/vocab.txt',
 '../models/distilbert-sentiment/added_tokens.json',
 '../models/distilbert-sentiment/tokenizer.json')

In [28]:
# tokenizer_config.json → tokenizer settings (lowercase, padding, etc.)
# special_tokens_map.json → special token mappings ([CLS], [PAD], etc.)
# vocab.txt → full vocabulary list (token → ID)
# tokenizer.json → compact full tokenizer config
# added_tokens.json → stores manually added tokens (if any)

# Make Sample Predictions

In [27]:
sample_text = "The flight was horrible and I missed my connection"
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, padding=True)
outputs = model(**inputs)
predicted_class = torch.argmax(outputs.logits).item()

label_map = {0: "Negative", 1: "Neutral", 2: "Positive"}
print("Predicted Sentiment:", label_map[predicted_class])

Predicted Sentiment: Negative
